# 🔍 Optimización S6 alpha_pf — S6 — Per-Client Macro-F1

## Federated Proactive Forest

**Estrategia:** Ranks trees by per-client macro-F1. Round-robin aggregation.

**Hiperparámetros:** `alpha_pf`, `t_max`, `local_weight`

**Datasets:** Letter, Optdigits, Spambase, Nursery, Sonar, Vowel

> ⚡ **Cada celda de dataset es independiente** — ejecuta solo la que necesites.

In [1]:
# ── Imports & Config ─────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().parent.parent.parent.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
import json
import optuna
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
N_CLIENTS = 5
N_TRIALS = 20
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / 'results' / 's6_alpha_optimization'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from src.domain.dataset.base_adapter import DatasetSplit
from src.application.orchestrators.fl_orchestrator import FLEXOrchestrator

strat_key = 'S6'
print(f'✅ Project root: {ROOT}')
print(f'✅ Strategy: {strat_key}')

c:\Users\Adrián Rodríguez\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest
✅ Strategy: S6


## Dataset: Letter

20,000 samples, 16 features, 26 classes (A-Z), numeric features

In [ ]:
# ── Load Letter ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'letter.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']

# Train/test split (80/20, stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='letter')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    alpha_pf = trial.suggest_float('alpha_pf', 0.1, 0.8, step=0.1); t_max = trial.suggest_int('t_max', 30, 150, step=10); local_weight = trial.suggest_float('local_weight', 0.0, 1.0, step=0.1)
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': alpha_pf,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {"strategy": "S6", "t_max": 100},
        'prediction': {'local_weight': local_weight, 'global_weight': 1.0 - local_weight},
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S6 en Letter... (20 trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n======================================================================")
print(f"📊 S6 — Letter — Resultados")
print(f"======================================================================")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'letter',
    'strategy': 'S6',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_letter_s6_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_letter_s6_results.json")

📊 Shape: (20000, 17)
✅ Train=16000, Test=4000, Feats=16, Classes=26
🚀 Optimizando S6 en Letter... (20 trials)


[I 2026-04-04 12:10:11,018] A new study created in memory with name: no-name-11d07530-050d-48fe-b39e-8095ff3caee4


## Dataset: Optdigits

5,620 samples, 64 features (8x8 pixel), 10 classes (0-9), numeric 0-16

In [ ]:
# ── Load Optdigits ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'optdigits.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']

# Train/test split (80/20, stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='optdigits')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    alpha_pf = trial.suggest_float('alpha_pf', 0.1, 0.8, step=0.1); t_max = trial.suggest_int('t_max', 30, 150, step=10); local_weight = trial.suggest_float('local_weight', 0.0, 1.0, step=0.1)
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': alpha_pf,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {"strategy": "S6", "t_max": 100},
        'prediction': {'local_weight': local_weight, 'global_weight': 1.0 - local_weight},
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S6 en Optdigits... (20 trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n======================================================================")
print(f"📊 S6 — Optdigits — Resultados")
print(f"======================================================================")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'optdigits',
    'strategy': 'S6',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_optdigits_s6_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_optdigits_s6_results.json")

## Dataset: Spambase

4,601 samples, 57 features (word frequencies), 2 classes (spam/ham)

In [ ]:
# ── Load Spambase ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'spambase.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']

# Train/test split (80/20, stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='spambase')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    alpha_pf = trial.suggest_float('alpha_pf', 0.1, 0.8, step=0.1); t_max = trial.suggest_int('t_max', 30, 150, step=10); local_weight = trial.suggest_float('local_weight', 0.0, 1.0, step=0.1)
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': alpha_pf,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {"strategy": "S6", "t_max": 100},
        'prediction': {'local_weight': local_weight, 'global_weight': 1.0 - local_weight},
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S6 en Spambase... (20 trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n======================================================================")
print(f"📊 S6 — Spambase — Resultados")
print(f"======================================================================")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'spambase',
    'strategy': 'S6',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_spambase_s6_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_spambase_s6_results.json")

## Dataset: Nursery

12,960 samples, 8 categorical features, 5 classes

In [ ]:
# ── Load Nursery ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'nursery.csv')
print(f'📊 Shape: {df.shape}')
# Encode categorical features
cat_cols = ['parents', 'has_nurs', 'form', 'children', 'housing', 'finance', 'social', 'health']
enc = OrdinalEncoder()
X = enc.fit_transform(df[cat_cols])
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']

# Train/test split (80/20, stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='nursery')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    alpha_pf = trial.suggest_float('alpha_pf', 0.1, 0.8, step=0.1); t_max = trial.suggest_int('t_max', 30, 150, step=10); local_weight = trial.suggest_float('local_weight', 0.0, 1.0, step=0.1)
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': alpha_pf,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {"strategy": "S6", "t_max": 100},
        'prediction': {'local_weight': local_weight, 'global_weight': 1.0 - local_weight},
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S6 en Nursery... (20 trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n======================================================================")
print(f"📊 S6 — Nursery — Resultados")
print(f"======================================================================")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'nursery',
    'strategy': 'S6',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_nursery_s6_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_nursery_s6_results.json")

## Dataset: Sonar

208 samples, 60 numeric features, 2 classes (Rock/Mine) — small dataset!

In [ ]:
# ── Load Sonar ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'sonar.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['Class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'Class']

# Train/test split (80/20, stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='sonar')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    alpha_pf = trial.suggest_float('alpha_pf', 0.1, 0.8, step=0.1); t_max = trial.suggest_int('t_max', 30, 150, step=10); local_weight = trial.suggest_float('local_weight', 0.0, 1.0, step=0.1)
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': alpha_pf,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {"strategy": "S6", "t_max": 100},
        'prediction': {'local_weight': local_weight, 'global_weight': 1.0 - local_weight},
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S6 en Sonar... (20 trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n======================================================================")
print(f"📊 S6 — Sonar — Resultados")
print(f"======================================================================")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'sonar',
    'strategy': 'S6',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_sonar_s6_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_sonar_s6_results.json")

## Dataset: Vowel

990 samples, 10 numeric features (drop 3 metadata cols), 11 classes

In [ ]:
# ── Load Vowel ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'vowel.csv')
print(f'📊 Shape: {df.shape}')
# Drop metadata columns
df = df.drop(columns=['Train or Test', 'Speaker Number', 'Sex'])
X = df.drop(columns=['Class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'Class']

# Train/test split (80/20, stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='vowel')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    alpha_pf = trial.suggest_float('alpha_pf', 0.1, 0.8, step=0.1); t_max = trial.suggest_int('t_max', 30, 150, step=10); local_weight = trial.suggest_float('local_weight', 0.0, 1.0, step=0.1)
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': alpha_pf,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {"strategy": "S6", "t_max": 100},
        'prediction': {'local_weight': local_weight, 'global_weight': 1.0 - local_weight},
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S6 en Vowel... (20 trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n======================================================================")
print(f"📊 S6 — Vowel — Resultados")
print(f"======================================================================")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'vowel',
    'strategy': 'S6',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_vowel_s6_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_vowel_s6_results.json")

## 📊 Resumen Global — Comparar todos los datasets

> ⚡ Ejecuta esta celda **después** de haber ejecutado todas las celdas de datasets.

In [ ]:
# ── Resumen Global (ejecutar después de todas las celdas) ─────────────
import glob

results = []
for fp in sorted((RESULTS_DIR).glob('s6_*_s6_results.json')):
    with open(fp) as f:
        r = json.load(f)
    row = {'dataset': r['dataset'], 'best_f1': round(r['best_macro_f1'], 4),
           'mean_f1': round(r['mean_macro_f1'], 4), 'std': round(r['std_macro_f1'], 4)}
    row.update(r['best_params'])
    results.append(row)

df_sum = pd.DataFrame(results)
print(f"\n{'='*90}")
print(f"📋 RESUMEN GLOBAL — S6 — Per-Client Macro-F1")
print(f"{'='*90}")
print(df_sum.to_string(index=False))

# Recommend alpha_pf
if 'alpha_pf' in df_sum.columns:
    rec = df_sum['alpha_pf'].median()
    print(f"\n🎯 alpha_pf recomendado (mediana): {{rec:.1f}}")
    print(f"   Rango: [{{df_sum['alpha_pf'].min()}} — {{df_sum['alpha_pf'].max()}}]")

df_sum.to_csv(RESULTS_DIR / f'summary_{strat_key.lower()}.csv', index=False)
print(f"\n✅ Resumen guardado: summary_{strat_key.lower()}.csv")